# Elder Mist Round 2 — the visual RL lab

This is the main lab.

You do **not** need Sea of Stars installed. We built a lightweight simulator around the strategically important structure of the second Elder Mist fight.

First you will play. Then rules will play. Then Q-learning will play.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Elder Mist as an RL environment</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">BATTLE STATE</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">POLICY</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">MOVE</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">BOSS REACTS</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">REWARD</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

repo = Path.cwd()
if repo.name == "notebooks": repo = repo.parent
sys.path.insert(0, str(repo))

from eldermist import ElderMistReplicaEnv, ACTIONS, ACTION_TO_ID
from eldermist.policies import sensible_rule_policy, explain_sensible_rule_policy
from eldermist.qlearning import train_q, q_policy, explain_q_choice
from eldermist.viz import plot_state

env = ElderMistReplicaEnv()
obs, info = env.reset(seed=7)
fig = plot_state(env, "Fresh episode after env.reset()")
plt.show()

## 1. The environment owns the battle

The environment contains Elder Mist HP, Sword state, Zale/Valere HP and MP, charged moves, locks, shields, timers, and turn logic.

The policy does **not** decide the damage rules.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">The environment implements consequences</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">TRUE BATTLE STATE</text><rect x="325.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="400.0" y="88" text-anchor="middle" font-size="15">env.step(action)</text><rect x="625.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="700.0" y="88" text-anchor="middle" font-size="15">NEXT BATTLE STATE</text>
<line x1="180.0" y1="82" x2="317.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="480.0" y1="82" x2="617.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
state = env.state
names = ["Boss HP","Sword HP","Zale HP","Valere HP"]
vals = [state.boss_hp/3995, state.sword_hp/800, state.zale_hp/300, state.valere_hp/300]

plt.figure(figsize=(8,4))
plt.bar(names, vals)
plt.ylim(0,1)
plt.ylabel("Fraction remaining")
plt.title("Some variables inside the true simulator state")
plt.show()

## 2. Observation is what gets handed out

Our environment returns a numeric observation vector. The world can contain more structure than that vector shows.

Later we deliberately change what counts as the **state representation** used by the policy.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">State ≠ observation</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">TRUE STATE</text><rect x="325.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="400.0" y="88" text-anchor="middle" font-size="15">OBSERVATION FUNCTION</text><rect x="625.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="700.0" y="88" text-anchor="middle" font-size="15">NUMERIC VECTOR</text>
<line x1="180.0" y1="82" x2="317.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="480.0" y1="82" x2="617.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
obs, _ = env.reset(seed=7)
labels = [
    "boss_hp","sword_hp","sword_active","zale_hp","valere_hp",
    "zale_mp","valere_mp","combo","actor","special","special_cd",
    "lock_total","sword_lock","blunt_lock","sun_lock","moon_lock"
]

plt.figure(figsize=(11,4))
plt.bar(labels, obs)
plt.xticks(rotation=65, ha="right")
plt.ylabel("Normalized value")
plt.title("The observation returned by env.reset()")
plt.tight_layout()
plt.show()

## 3. Play it yourself — **you are the policy**

Click a legal move.

The simulator will apply the move, progress timers, possibly trigger Counter Slash or a charged move, and return the next observation and reward.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Manual policy</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">OBSERVATION</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">YOUR BRAIN = π</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">CLICK ACTION</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">env.step()</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">NEW STATE</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
manual = ElderMistReplicaEnv(reward_mode="shaped")
manual.reset(seed=11)

out_widget = widgets.Output()
buttons = {a: widgets.Button(description=a.replace("_"," "), layout=widgets.Layout(width="175px"))
           for a in ACTIONS}
reset_button = widgets.Button(description="RESET FIGHT", button_style="warning")

def redraw_manual():
    with out_widget:
        clear_output(wait=True)
        fig = plot_state(manual, "YOU are choosing the next action")
        display(fig)
        plt.close(fig)
        print("Valid:", manual.valid_actions())
        print("Last:", manual.last_event)
    valid = set(manual.valid_actions())
    for name, b in buttons.items():
        b.disabled = name not in valid or manual._done()

def make_cb(name):
    def _cb(_):
        _, r, term, trunc, _ = manual.step(ACTION_TO_ID[name])
        redraw_manual()
        with out_widget:
            print(f"Reward from last transition: {r:+.3f}")
    return _cb

for name, b in buttons.items():
    b.on_click(make_cb(name))

def do_reset(_):
    manual.reset(seed=11)
    redraw_manual()
reset_button.on_click(do_reset)

display(widgets.VBox([
    widgets.HBox(list(buttons.values())[:4]),
    widgets.HBox(list(buttons.values())[4:]),
    reset_button,
    out_widget
]))
redraw_manual()

## 4. Why did a move cause that result?

The policy chooses only the **action**.

The environment's transition logic determines damage, lock breaking, counters, boss turns, Sword revival, and charged attacks.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Action is not consequence</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">ACTION: SUNBALL</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">TRANSITION RULES</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">SWORD HP ↓</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">BOSS HP ↓</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">NEW OBS</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
demo = ElderMistReplicaEnv()
demo.reset(seed=2)
before = demo.state.sword_hp
_, reward, _, _, _ = demo.step(ACTION_TO_ID["sunball_sword"])
after = demo.state.sword_hp

plt.figure(figsize=(7,4))
plt.bar(["Sword HP before","Sword HP after"], [before, after])
plt.ylabel("HP")
plt.title(f"One action → environment transition | reward={reward:+.3f}")
plt.show()

print(demo.last_event)

## 5. A rule policy is still a policy

This baseline contains readable `if/else` logic.

If it chooses Sunball, we can explain the move because **we explicitly wrote the reason into the policy**.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Policy without learning</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">STATE</text><rect x="325.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="400.0" y="88" text-anchor="middle" font-size="15">IF/ELSE RULES</text><rect x="625.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="700.0" y="88" text-anchor="middle" font-size="15">MOVE</text>
<line x1="180.0" y1="82" x2="317.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="480.0" y1="82" x2="617.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
demo = ElderMistReplicaEnv()
demo.reset(seed=8)
move, reason = explain_sensible_rule_policy(demo)

print("Chosen move:", move)
print("WHY:", reason)

choices = demo.valid_actions()
scores = [1 if x == move else 0 for x in choices]
plt.figure(figsize=(9,4))
plt.bar(choices, scores)
plt.xticks(rotation=25)
plt.ylabel("Rule selects?")
plt.title("Readable rule policy: exactly one branch fires")
plt.show()

## 6. Compare random behavior with a sensible hand-written policy

This is **evaluation**, not learning.

We run many fresh episodes and measure win rate and Counter Slashes.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Evaluate behavior</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">POLICY</text><rect x="325.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="400.0" y="88" text-anchor="middle" font-size="15">100 FRESH FIGHTS</text><rect x="625.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="700.0" y="88" text-anchor="middle" font-size="15">WIN RATE / COUNTERS</text>
<line x1="180.0" y1="82" x2="317.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="480.0" y1="82" x2="617.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
def evaluate_policy(policy, seeds):
    rows = []
    for seed in seeds:
        e = ElderMistReplicaEnv()
        o, _ = e.reset(seed=seed)
        while True:
            a = policy(e, o)
            o, r, term, trunc, _ = e.step(a)
            if term or trunc:
                break
        rows.append({
            "win": int(e.state.boss_hp <= 0),
            "counter_slashes": e.state.counter_slashes,
            "turns": e.state.turn
        })
    return pd.DataFrame(rows)

def random_policy(e, o):
    return ACTION_TO_ID[np.random.choice(e.valid_actions())]

rand_df = evaluate_policy(random_policy, range(100))
rule_df = evaluate_policy(sensible_rule_policy, range(100))

plt.figure(figsize=(7,4))
plt.bar(["Random","Rule policy"], [rand_df.win.mean(), rule_df.win.mean()])
plt.ylim(0,1)
plt.ylabel("Win rate")
plt.title("Policy quality before any Q-learning")
plt.show()

print("Average Counter Slashes — random:", rand_df.counter_slashes.mean())
print("Average Counter Slashes — rule:", rule_df.counter_slashes.mean())

## 7. Q-learning: learning lives in the Q-table

For every represented state and legal action, store an estimate:

`Q(state, action) ≈ expected future return`.

Experience changes those numbers.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">The learner&#x27;s memory</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">STATE</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">ACTION</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">REWARD</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">NEXT STATE</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">Q UPDATE</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
q, train_rows = train_q(
    lambda: ElderMistReplicaEnv(representation="full", reward_mode="shaped"),
    episodes=8000,
    seed=42
)
train_df = pd.DataFrame(train_rows)

plt.figure(figsize=(9,4))
plt.plot(train_df["episode"], train_df["return"].rolling(200, min_periods=1).mean())
plt.xlabel("Training episode")
plt.ylabel("Rolling return")
plt.title("8,000 headless Elder Mist fights: Q-learning curve")
plt.show()

## 8. Why did the learned policy choose *this* move?

Unlike the rule policy, there may be no English `if Sword active → Sunball` rule.

The Q-policy chooses the legal action with the largest learned Q-value for the current represented state.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Q-policy decision</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">CURRENT REPRESENTATION</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">LOOK UP Q VALUES</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">ARGMAX</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">ACTION</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
inspect_env = ElderMistReplicaEnv(representation="full")
inspect_env.reset(seed=123)
exp = explain_q_choice(q, inspect_env)
qdf = pd.DataFrame(exp["ranked_actions"])

print("Represented state:", exp["represented_state"])
print("Chosen:", exp["chosen_action"])
display(qdf)

plt.figure(figsize=(9,4))
plt.bar(qdf["action"], qdf["q_value"])
plt.xticks(rotation=25)
plt.ylabel("Learned Q-value")
plt.title("Why this Q-policy chose its move")
plt.show()

## 9. Watch one learned-policy fight unfold

Now freeze learning and act greedily from the Q-table.

The following plot shows Elder Mist HP over the sequence of actions.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Evaluation rollout</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">LEARNED Q TABLE</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">GREEDY ACTION</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">BATTLE</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">REPEAT</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
e = ElderMistReplicaEnv(representation="full")
o, _ = e.reset(seed=321)
rows = []

while True:
    a = q_policy(q, e)
    action_name = ACTIONS[a]
    o, r, term, trunc, _ = e.step(a)
    rows.append({
        "turn": e.state.turn,
        "boss_hp": e.state.boss_hp,
        "sword_hp": e.state.sword_hp,
        "counter_slashes": e.state.counter_slashes,
        "action": action_name
    })
    if term or trunc:
        break

roll = pd.DataFrame(rows)

plt.figure(figsize=(10,4))
plt.plot(roll["turn"], roll["boss_hp"], marker="o")
plt.xlabel("Player action")
plt.ylabel("Elder Mist HP")
plt.title(f"One learned-policy fight | win={e.state.boss_hp <= 0}")
plt.show()

display(roll.head(20))

## 10. State representation experiment

Now hold the environment and learner fixed.

Only change what the policy is allowed to distinguish:

- FULL
- NO SWORD
- NO LOCKS
- MINIMAL

This is the exact logic behind testing screenshot vs accessibility tree vs history.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Controlled ablation</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">SAME WORLD</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">DIFFERENT REPRESENTATION</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">SAME LEARNER</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">COMPARE</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
reps = ["full","no_sword","no_locks","minimal"]
summary = []

for rep in reps:
    qrep, _ = train_q(
        lambda rep=rep: ElderMistReplicaEnv(representation=rep, reward_mode="shaped"),
        episodes=5000,
        seed=100
    )
    wins = []
    counters = []
    for seed in range(10000,10150):
        er = ElderMistReplicaEnv(representation=rep, reward_mode="shaped")
        o,_ = er.reset(seed=seed)
        while True:
            a = q_policy(qrep, er)
            o,r,term,trunc,_ = er.step(a)
            if term or trunc: break
        wins.append(int(er.state.boss_hp <= 0))
        counters.append(er.state.counter_slashes)
    summary.append((rep, np.mean(wins), np.mean(counters)))

summary_df = pd.DataFrame(summary, columns=["representation","win_rate","avg_counters"])
display(summary_df)

plt.figure(figsize=(8,4))
plt.bar(summary_df["representation"], summary_df["win_rate"])
plt.ylim(0,1)
plt.ylabel("Held-out win rate")
plt.title("State representation changes downstream behavior")
plt.show()

## 11. The research interpretation

If hiding Sword state hurts performance, the reason is not “less telemetry sounds bad.”

The reason is that two strategically different worlds become indistinguishable to the policy.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">State aliasing</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">WORLD A: SWORD UP</text><rect x="250.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="325.0" y="88" text-anchor="middle" font-size="15">SAME REPRESENTATION?</text><rect x="475.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="550.0" y="88" text-anchor="middle" font-size="15">WORLD B: SWORD DOWN</text><rect x="700.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="775.0" y="88" text-anchor="middle" font-size="15">BAD ALIASING</text>
<line x1="180.0" y1="82" x2="242.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="405.0" y1="82" x2="467.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="630.0" y1="82" x2="692.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>


In [ ]:
rep_size = {"full":13, "no_sword":11, "no_locks":10, "minimal":4}
summary_df["fields"] = summary_df["representation"].map(rep_size)

plt.figure(figsize=(8,5))
plt.scatter(summary_df["fields"], summary_df["win_rate"], s=110)
for _, r in summary_df.iterrows():
    plt.text(r["fields"]+0.15, r["win_rate"], r["representation"])
plt.xlabel("Fields retained in representation")
plt.ylabel("Held-out win rate")
plt.ylim(0,1)
plt.title("Representation cost vs performance")
plt.show()

## Once this clicks → POMDP

So far the agent is handed a representation each turn.

Next we deliberately hide truth and give noisy evidence. Then the agent needs **history** and a **belief** about hidden state.


<div style="margin:12px 0">
<svg width="100%" viewBox="0 0 900 150" role="img">
<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto">
<path d="M0,0 L8,4 L0,8 z" fill="black"/></marker></defs>
<text x="450.0" y="28" text-anchor="middle" font-size="20" font-weight="bold">Next: partial observability</text>
<rect x="25.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="100.0" y="88" text-anchor="middle" font-size="15">HIDDEN TRUE STATE</text><rect x="205.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="280.0" y="88" text-anchor="middle" font-size="15">NOISY OBSERVATIONS</text><rect x="385.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="460.0" y="88" text-anchor="middle" font-size="15">HISTORY</text><rect x="565.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="640.0" y="88" text-anchor="middle" font-size="15">BELIEF</text><rect x="745.0" y="55" rx="12" width="150.0" height="55" fill="white" stroke="black" stroke-width="2"/><text x="820.0" y="88" text-anchor="middle" font-size="15">POLICY</text>
<line x1="180.0" y1="82" x2="197.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="360.0" y1="82" x2="377.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="540.0" y1="82" x2="557.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/><line x1="720.0" y1="82" x2="737.0" y2="82" stroke="black" stroke-width="2" marker-end="url(#arrow)"/>
</svg>
</div>
